# Data Analytics — Exercise 7.3: Pandas DataFrames & Merging
## Marenza Santarin | 5/19/2026

This notebook analyzes GlobalTrade Ltd.'s orders and shipments data. It loads both CSV files, merges them together, and identifies shipped, unshipped, and unmatched shipment records.

In [2]:
import pandas as pd
import os

print(os.listdir('.'))
# Checking to see if the 2 .csv files are showing in the same folder

['DA_Exercise_7_1.ipynb', 'DA_Exercise_7_2.ipynb', 'DA_Exercise_7_3.ipynb', 'orders.csv', 'shipments.csv']


In [7]:
# Loading the .csv files
# df_orders stores the order table
# df_shipments stores the shipment table
df_orders = pd.read_csv('orders.csv')
df_shipments = pd.read_csv('shipments.csv')

# Show the top 5 rows
# Show the number of rows and columns
# Show the data types
# Show and add any missing values
print(df_orders.head())
print()
print(df_orders.shape)
print()
print(df_orders.dtypes)
print()
print(df_orders.isnull().sum())
print()


print(df_shipments.head())
print()
print(df_shipments.shape)
print()
print(df_shipments.dtypes)
print()
print(df_shipments.isnull().sum())

   order_id customer   product  quantity  unit_price
0       101     Chen    Laptop         2      899.99
1       102    Patel    Tablet         5      499.50
2       103   Müller   Monitor         1      349.00
3       104  Okonkwo  Keyboard        10       79.99
4       105   Santos   Headset         3      149.99

(10, 5)

order_id        int64
customer          str
product           str
quantity        int64
unit_price    float64
dtype: object

order_id      0
customer      0
product       0
quantity      0
unit_price    0
dtype: int64

   order_id  ship_date carrier      status
0       101  1/10/2024     DHL   Delivered
1       102  1/12/2024   FedEx   Delivered
2       103  1/13/2024     DHL  In Transit
3       104  1/14/2024     UPS   Delivered
4       105  1/15/2024   FedEx     Delayed

(8, 4)

order_id     int64
ship_date      str
carrier        str
status         str
dtype: object

order_id     0
ship_date    0
carrier      0
status       0
dtype: int64


## Inspect and Clean Both DataFrames

In [15]:
# This will create a new column called 'order_value' and multiplies each row's quantity by that row's unit price
df_orders['order_value'] = df_orders['quantity'] * df_orders['unit_price']

# This will only print the columns specified
print(df_orders[['order_id', 'customer', 'product', 'order_value']])
print() 
# This will automatically give a summary for all numeric values
print(df_orders.describe())

# This will change the date from regular text to date format
df_shipments['ship_date'] = pd.to_datetime(df_shipments['ship_date'])
print()
print(df_shipments.dtypes)

# This will add up all order values
total_order_value = df_orders['order_value'].sum()

# Prints the total with commas and 2 decimal places
print()
print(f"Total order value: ${total_order_value:,.2f}")

   order_id  customer          product  order_value
0       101      Chen           Laptop      1799.98
1       102     Patel           Tablet      2497.50
2       103    Müller          Monitor       349.00
3       104   Okonkwo         Keyboard       799.90
4       105    Santos          Headset       449.97
5       106   Larsson           Webcam       359.96
6       107       Kim            Mouse       239.92
7       108   O'Brien  Docking Station       199.00
8       109     Rossi              SSD       719.94
9       110  Nakamura          USB Hub        79.98

        order_id   quantity  unit_price  order_value
count   10.00000  10.000000   10.000000    10.000000
mean   105.50000   4.200000  245.743000   749.515000
std      3.02765   3.047768  273.023774   787.066644
min    101.00000   1.000000   29.990000    79.980000
25%    103.25000   2.000000   82.490000   267.190000
50%    105.50000   3.500000  134.990000   404.965000
75%    107.75000   5.750000  311.500000   779.910000
max

## Perform All Four Merge Types

In [18]:
# pd.merge() connects tables using order_id
# inner keeps only matching order IDs in both files
# left keeps all orders, even if they do not have shipment info
# right keeps all shipments, even if they do not have an order record
# outer keeps everything from both files
df_inner = pd.merge(df_orders, df_shipments, on='order_id', how='inner')
print("INNER MERGE")
print(df_inner)

df_left = pd.merge(df_orders, df_shipments, on='order_id', how='left')
print("\nLEFT MERGE")
print(df_left)

df_right = pd.merge(df_orders, df_shipments, on='order_id', how='right')
print("\nRIGHT MERGE")
print(df_right)

df_outer = pd.merge(df_orders, df_shipments, on='order_id', how='outer')
print("\nOUTER MERGE")
print(df_outer)

print(df_outer.shape)
print(df_outer.isnull().sum().sum())

INNER MERGE
   order_id customer   product  quantity  unit_price  order_value  ship_date  \
0       101     Chen    Laptop         2      899.99      1799.98 2024-01-10   
1       102    Patel    Tablet         5      499.50      2497.50 2024-01-12   
2       103   Müller   Monitor         1      349.00       349.00 2024-01-13   
3       104  Okonkwo  Keyboard        10       79.99       799.90 2024-01-14   
4       105   Santos   Headset         3      149.99       449.97 2024-01-15   
5       106  Larsson    Webcam         4       89.99       359.96 2024-01-16   
6       107      Kim     Mouse         8       29.99       239.92 2024-01-18   

  carrier      status  
0     DHL   Delivered  
1   FedEx   Delivered  
2     DHL  In Transit  
3     UPS   Delivered  
4   FedEx     Delayed  
5     UPS   Delivered  
6     DHL  In Transit  

LEFT MERGE
   order_id  customer          product  quantity  unit_price  order_value  \
0       101      Chen           Laptop         2      899.99      

## Add Shipped Flag and Calculate Fulfilment Rate

In [25]:
# This creates a new column called shipped inside df_left
# It checks the carrier column
# If there is a carrier name like DHL, FedEx, or UPS, it stores True
# If the carrier is missing/NaN, it stores False
df_left['shipped'] = df_left['carrier'].notna()

# Prints only the selected columns
print(df_left[['order_id', 'customer', 'order_value', 'shipped']])

# Counts how many true and false values are in the shipped column
print(df_left['shipped'].value_counts())

# Counts how many orders were shipped
# Since True acts like 1 and False acts like 0, .sum() adds up the True values
shipped_count = df_left['shipped'].sum()

# Counts how many total orders are left in df_left
total_orders = len(df_left)

# Calculates the percentage of orders that shipped
fulfilment_rate = shipped_count / total_orders * 100

# Prints the fulfilment rate
print(f"Fulfilment rate: {fulfilment_rate:.0f}%")

   order_id  customer  order_value  shipped
0       101      Chen      1799.98     True
1       102     Patel      2497.50     True
2       103    Müller       349.00     True
3       104   Okonkwo       799.90     True
4       105    Santos       449.97     True
5       106   Larsson       359.96     True
6       107       Kim       239.92     True
7       108   O'Brien       199.00    False
8       109     Rossi       719.94    False
9       110  Nakamura        79.98    False
shipped
True     7
False    3
Name: count, dtype: int64
Fulfilment rate: 70%


## GroupBy Summary

In [27]:
# This groups the matched orders by shipment status, like Delivered, Delayed, or In Transit
# Then it looks at order_value and calculates the total, average, and count for each status
status_summary = df_inner.groupby('status')['order_value'].agg(['sum', 'mean', 'count'])
print(status_summary)

# This groups the matched orders by carrier, like DHL, FedEx, and UPS
# Then it adds up the total order value for each carrier and sorts them from highest to lowest
carrier_totals = df_inner.groupby('carrier')['order_value'].sum().sort_values(ascending=False)
print(carrier_totals)

# This prints the carrier at the top of the sorted list
# Since it was sorted from highest to lowest, index [0] gives the highest carrier
print(f"Top carrier: {carrier_totals.index[0]}")

# This groups the matched orders by status again, but this time it adds up the quantity column
# This tells how many total units are Delivered, Delayed, or In Transit
units_per_status = df_inner.groupby('status')['quantity'].sum()
print(units_per_status)

                sum      mean  count
status                              
Delayed      449.97   449.970      1
Delivered   5457.34  1364.335      4
In Transit   588.92   294.460      2
carrier
FedEx    2947.47
DHL      2388.90
UPS      1159.86
Name: order_value, dtype: float64
Top carrier: FedEx
status
Delayed        3
Delivered     21
In Transit     9
Name: quantity, dtype: int64


## Summary

The fulfilment rate is 70%, meaning 7 out of 10 orders have shipment records. 
Orders 108, 109, and 110 are unshipped because they appear in the orders file but not in the shipments file. 
Order 111 is an orphan shipment because it appears in the shipments file but has no matching order record.